In [5]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter
from datasets import load_dataset

# Set plot style for better readability
plt.style.use('seaborn-v0_8-whitegrid')
%matplotlib inline

Matplotlib is building the font cache; this may take a moment.
c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# Load the ARF dataset (synthetic relations subset)
print("Loading dataset... This may take a moment on the first run.")
dataset = load_dataset("Despina/project_gutenberg", "synthetic_relations_in_fiction_books")

# Inspect the structure
print(f"Dataset splits: {dataset.keys()}")
print(f"Number of rows in 'train' split: {len(dataset['train']):,}")

Loading dataset... This may take a moment on the first run.


c:\Users\baaqa\Downloads\Narrative Intelligence\narrative-intelligence\.venv\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\baaqa\.cache\huggingface\hub\datasets--Despina--project_gutenberg. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Generating train split: 100%|██████████| 95476/95476 [00:01<00:0

Dataset splits: dict_keys(['train'])
Number of rows in 'train' split: 95,476


# Dataset Reconnaissance: 
### we need to know exactly what fields ARF gives us: what counts as an entity, what a relationship record looks like, whether there's any temporal signal already, and how dirty the data is.

In [8]:
# Look at a single raw example before assuming anything about structure
sample = dataset["train"][0]
print(type(sample))
for key, value in sample.items():
    print(f"{key!r}: {type(value)} -> {value}")

<class 'dict'>
'book_id': <class 'str'> -> 106
'title': <class 'str'> -> Jungle Tales of Tarzan
'author': <class 'str'> -> Edgar Rice Burroughs
'author_gender': <class 'str'> -> male
'author_birth_year': <class 'str'> -> 1875
'author_death_year': <class 'str'> -> 1950
'release_date': <class 'str'> -> Feb 1, 1994
'pg_subjects': <class 'str'> -> ['Tarzan (Fictitious character) -- Fiction', 'Africa -- Fiction', 'Fantasy fiction', 'Jungles -- Fiction', 'Adventure stories', 'Apes -- Fiction']
'topics': <class 'str'> -> ['fantasy fiction', 'stories', 'adventure stories', 'fiction']
'chunk_id': <class 'str'> -> 0
'chunk': <class 'str'> -> ***  JUNGLE TALES OF TARZAN ***

[Illustration]




Jungle Tales of Tarzan

by Edgar Rice Burroughs




Contents

 CHAPTER I. Tarzan's First Love  CHAPTER II. The Capture of Tarzan  CHAPTER III. The Fight for the Balu  CHAPTER IV. The God of Tarzan  CHAPTER V. Tarzan and the Black Boy  CHAPTER VI. The Witch-Doctor Seeks Vengeance  CHAPTER VII.
'relations': <

In [9]:
print("Features schema:")
print(dataset["train"].features)

Features schema:
{'book_id': Value('string'), 'title': Value('string'), 'author': Value('string'), 'author_gender': Value('string'), 'author_birth_year': Value('string'), 'author_death_year': Value('string'), 'release_date': Value('string'), 'pg_subjects': Value('string'), 'topics': Value('string'), 'chunk_id': Value('string'), 'chunk': Value('string'), 'relations': Value('string')}


In [10]:
df = dataset["train"].to_pandas()
print(df.shape)
df.head(3)

(95476, 12)


,book_id,title,author,author_gender,author_birth_year,author_death_year,release_date,pg_subjects,topics,chunk_id,chunk,relations
0,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",0,*** JUNGLE TALES OF TARZAN ***\r\n\r\n[Illust...,[]
1,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",1,The Witch-Doctor Seeks Vengeance CHAPTER VII....,[]
2,106,Jungle Tales of Tarzan,Edgar Rice Burroughs,male,1875,1950,"Feb 1, 1994","['Tarzan (Fictitious character) -- Fiction', '...","['fantasy fiction', 'stories', 'adventure stor...",10,It is true that Taug was no longer the frolics...,"[{'entity1': 'Taug', 'entity2': 'Tarzan', 'ent..."


In [11]:
# Entity-related columns: which fields identify characters/entities?
print("Columns:", df.columns.tolist())

# Null counts — dirty data means canonicalization work in Week 1
print("\nNull counts:")
print(df.isnull().sum())

# Any duplicate rows?
print(f"\nDuplicate rows: {df.duplicated().sum()}")

Columns: ['book_id', 'title', 'author', 'author_gender', 'author_birth_year', 'author_death_year', 'release_date', 'pg_subjects', 'topics', 'chunk_id', 'chunk', 'relations']

Null counts:
book_id              0
title                1
author               1
author_gender        1
author_birth_year    1
author_death_year    1
release_date         1
pg_subjects          1
topics               1
chunk_id             1
chunk                1
relations            1
dtype: int64

Duplicate rows: 0


In [12]:
# 1. Does a relationship type / label column exist, and what values does it take?
# (replace 'relation' with whatever the actual column is called once you see Cell 2/3 output)
if "relation" in df.columns:
    print(df["relation"].value_counts())

# 2. Is there ANY temporal/ordering signal already in the data?
#    e.g. chapter number, sentence position, book order, timestamps
temporal_candidates = [c for c in df.columns if any(
    kw in c.lower() for kw in ["time", "order", "chapter", "position", "sequence", "date"]
)]
print("Possible temporal columns:", temporal_candidates)

Possible temporal columns: ['release_date']


In [13]:
import ast

row = df.iloc[2]
relations = ast.literal_eval(row["relations"])
print(f"Number of relations in this chunk: {len(relations)}")
for r in relations:
    print(r)

Number of relations in this chunk: 2
{'entity1': 'Taug', 'entity2': 'Tarzan', 'entity1Type': 'PER', 'entity2Type': 'PER', 'relation': 'companion_of'}
{'entity1': 'Taug', 'entity2': 'Teeka', 'entity1Type': 'PER', 'entity2Type': 'PER', 'relation': 'companion_of'}


In [14]:
book_106 = df[df["book_id"] == "106"].copy()
book_106["chunk_id"] = book_106["chunk_id"].astype(int)
book_106 = book_106.sort_values("chunk_id")
print(f"Chunks for book 106: {len(book_106)}")
print(f"chunk_id range: {book_106['chunk_id'].min()} to {book_106['chunk_id'].max()}")
print(f"Is it contiguous (no gaps)? {book_106['chunk_id'].tolist() == list(range(book_106['chunk_id'].min(), book_106['chunk_id'].max()+1))}")

Chunks for book 106: 883
chunk_id range: 0 to 882
Is it contiguous (no gaps)? True


In [19]:
import ast

def safe_parse_relations(raw: object) -> list | None:
    """Parse a stringified relations list.

    Returns [] for genuinely empty relations, a list of dicts for valid data,
    or None if the string is malformed (can't be parsed at all).
    """
    if not isinstance(raw, str):
        return None  # covers NaN / non-string cells
    try:
        return ast.literal_eval(raw)
    except (ValueError, SyntaxError):
        return None

df["relations_parsed"] = df["relations"].apply(safe_parse_relations)

malformed = df[df["relations_parsed"].isnull()]
print(f"Malformed/unparseable rows: {len(malformed)} / {len(df):,}")
print(malformed[["book_id", "chunk_id", "relations"]])

Malformed/unparseable rows: 1 / 95,476
               book_id chunk_id relations
95475  ompanion_of'}]"      NaN       NaN


In [20]:
valid = df[df["relations_parsed"].notnull()].copy()
valid["num_relations"] = valid["relations_parsed"].apply(len)

print(f"Valid rows: {len(valid):,} / {len(df):,}")
print(f"Chunks with >=1 relation: {(valid['num_relations'] > 0).sum():,}")
print(f"Total relation instances: {valid['num_relations'].sum():,}")
print(f"Unique books: {valid['book_id'].nunique()}")

Valid rows: 95,475 / 95,476
Chunks with >=1 relation: 60,245
Total relation instances: 128,331
Unique books: 96


In [21]:
def is_contiguous(group: pd.DataFrame) -> bool:
    ids = sorted(group["chunk_id"].astype(int))
    return ids == list(range(ids[0], ids[-1] + 1))

# sample 20 random books, not just book 106 — one example isn't proof
sample_books = valid["book_id"].dropna().unique()
import random
random.seed(42)
sample = random.sample(list(sample_books), min(20, len(sample_books)))

results = {
    bid: is_contiguous(valid[valid["book_id"] == bid])
    for bid in sample
}
print(results)
print(f"\nContiguous in {sum(results.values())}/{len(results)} sampled books")

{'73548': True, '21299': True, '12807': True, '9909': True, '36684': True, '34025': True, '32543': True, '23060': True, '18873': True, '77': True, '619': True, '16630': True, '6941': True, '5111': False, '1329': True, '31858': True, '3322': True, '5658': True, '70653': True, '64264': True}

Contiguous in 19/20 sampled books


In [22]:
book_5111 = valid[valid["book_id"] == "5111"].copy()
book_5111["chunk_id"] = book_5111["chunk_id"].astype(int)
book_5111 = book_5111.sort_values("chunk_id")

ids = book_5111["chunk_id"].tolist()
gaps = [(ids[i], ids[i+1]) for i in range(len(ids)-1) if ids[i+1] - ids[i] > 1]

print(f"Total chunks: {len(ids)}, range {ids[0]}–{ids[-1]}")
print(f"Gaps found: {gaps}")

Total chunks: 506, range 0–506
Gaps found: [(50, 52)]


In [25]:
# Save the cleaned relations dataframe for use in Week 1 graph-building work,
# so later notebooks/scripts don't need to re-download + re-parse from scratch.
valid.to_parquet("../data/arf_chunks_parsed.parquet", index=False)
print("Saved cleaned dataset to data/arf_chunks_parsed.parquet")

Saved cleaned dataset to data/arf_chunks_parsed.parquet
